# Merge Pentacam school folders

Merges folders named `SCHOOLNAME_DD-MM-YYYY` — found inside `Pentacam 1 SSP HYD`, `Pentacam 2 SSP HYD`, `Pentacam 3 SSP HYD` — into **one combined folder per school name**, directly under the root folder.

Example:
```
C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\AHAD HIGH SCHOOL_25-03-2026
C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\AHAD HIGH SCHOOL_26-03-2026
C:\HYD Extracted Pentacam\Pentacam 2 SSP HYD\AHAD HIGH SCHOOL_02-04-2026
```
all get merged into:
```
C:\HYD Extracted Pentacam\AHAD HIGH SCHOOL\...
```

**Because the same filenames repeat across every date-folder for a school**, every copied file has its source folder's date appended to its name, e.g.:
```
Patient1_25-03-2026.pdf
Patient1_26-03-2026.pdf
Patient1_02-04-2026.pdf
```
so nothing is ever overwritten, and you can always tell which date-folder a file came from.

Run the cells in order. **The first real run should be a dry run** (default) so you can check the plan before anything is copied.

In [1]:
import pandas as pd

In [5]:
df1 = pd.read_excel(r"C:\Users\shivam.prajapati\Downloads\All_Schools_Combined.xlsx")
df2 = pd.read_excel(r"C:\Users\shivam.prajapati\Downloads\All_Schools_Combined (1).xlsx")
df3 = pd.read_excel(r"C:\Users\shivam.prajapati\Downloads\All_Schools_Combined (2).xlsx")

In [6]:
df = pd.concat([df1, df2,df3], ignore_index=True)
df.shape

(38189, 28)

In [7]:
df.to_excel(r"C:\Users\shivam.prajapati\Downloads\All_Schools_Combined_HYD.xlsx", index=False)

In [1]:
import re
import shutil
from pathlib import Path

# ---------------- Parameters ----------------
ROOT_PATH = r"C:\HYD Extracted Pentacam"

DRY_RUN = True      # True = just print the plan, no files touched
MOVE = False        # False = copy files (originals kept). True = move files (originals removed)

ALWAYS_SUFFIX_DATE = True  # True = every file gets its source folder's date appended (recommended,
                            # since duplicate filenames across date-folders are expected here).
                            # False = only files that collide with an existing name get suffixed.
# ----------------------------------------------

In [2]:
DATE_FOLDER_PATTERN = re.compile(r'^(?P<school>.+)_(?P<date>\d{2}-\d{2}-\d{4})$')


def find_pentacam_folders(root: Path):
    """Find 'Pentacam X SSP HYD' folders directly under root."""
    pattern = re.compile(r'^Pentacam \d+ SSP HYD$')
    return [p for p in root.iterdir() if p.is_dir() and pattern.match(p.name)]


def group_date_folders(pentacam_folders):
    """Group school date-folders by school name. Returns (grouped, skipped)."""
    grouped = {}
    skipped = []

    for pf in pentacam_folders:
        for folder in pf.iterdir():
            if not folder.is_dir():
                continue
            m = DATE_FOLDER_PATTERN.match(folder.name)
            if m:
                school = m.group('school').strip()
                date = m.group('date')
                grouped.setdefault(school, []).append((folder, date))
            else:
                skipped.append(folder)

    return grouped, skipped


def unique_dest_path(dest_path: Path):
    """If dest_path exists, add a numeric counter until it doesn't."""
    if not dest_path.exists():
        return dest_path
    stem, suffix, parent = dest_path.stem, dest_path.suffix, dest_path.parent
    counter = 1
    candidate = dest_path
    while candidate.exists():
        candidate = parent / f"{stem}_{counter}{suffix}"
        counter += 1
    return candidate


def build_dest_path(target_folder: Path, relative_path: Path, date: str, always_suffix: bool):
    dest_path = target_folder / relative_path

    needs_suffix = always_suffix or dest_path.exists()
    if needs_suffix:
        stem = dest_path.stem
        suffix = dest_path.suffix
        dest_path = dest_path.parent / f"{stem}_{date}{suffix}"
        dest_path = unique_dest_path(dest_path)

    return dest_path


def merge_school_folders(root_path: str, dry_run: bool = True, move: bool = False, always_suffix_date: bool = True):
    root = Path(root_path)
    if not root.exists():
        raise FileNotFoundError(f"Root path not found: {root}")

    pentacam_folders = find_pentacam_folders(root)
    if not pentacam_folders:
        print(f"WARNING: No 'Pentacam X SSP HYD' folders found directly under {root}.")

    grouped, skipped = group_date_folders(pentacam_folders)

    if skipped:
        print("\nSkipping folders that don't match 'SCHOOLNAME_DD-MM-YYYY':")
        for s in skipped:
            print(f"  {s}")

    print(f"\nFound {len(grouped)} unique school(s).")
    if dry_run:
        print("*** DRY RUN MODE - no files will be copied/moved. Set DRY_RUN = False to apply. ***\n")

    verb = "MOVE" if move else "COPY"

    for school in sorted(grouped.keys()):
        target_folder = root / school
        print(f"\n=== {school} ===")
        print(f"Target: {target_folder}")

        if not dry_run:
            target_folder.mkdir(parents=True, exist_ok=True)

        for source_folder, date in grouped[school]:
            print(f"  Source: {source_folder}")

            files = [f for f in source_folder.rglob('*') if f.is_file()]
            for file in files:
                relative_path = file.relative_to(source_folder)
                dest_path = build_dest_path(target_folder, relative_path, date, always_suffix_date)

                print(f"    [{verb}] {relative_path} -> {dest_path}")

                if not dry_run:
                    dest_path.parent.mkdir(parents=True, exist_ok=True)
                    if move:
                        shutil.move(str(file), str(dest_path))
                    else:
                        shutil.copy2(str(file), str(dest_path))

    print("\nDone.")
    if dry_run:
        print("This was a dry run. Set DRY_RUN = False (in the parameters cell) to actually copy/move files.")

In [4]:
merge_school_folders(ROOT_PATH, dry_run=False, move=MOVE, always_suffix_date=ALWAYS_SUFFIX_DATE)


Skipping folders that don't match 'SCHOOLNAME_DD-MM-YYYY':
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Alif high school _ 05-01-2026
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Aziz Nagar 14-7-2025 pentacam
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Aziz Nagar 15-07-2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Bhavani Model High School 17-11-2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Chaitanya Pragathi High School 08-12-2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Donald International School 24-03-2026
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Farhat high school 16-12-2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Hydershakote 16-07-2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\HYDERSHAKOTE 17=07-2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Imaan global islamic school 15-12-2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\Infant Jesus_30.10.2025
  C:\HYD Extracted Pentacam\Pentacam 1 SSP HYD\krishnaveni talent school hyderguda